In [ ]:
import os
import random
import time
from typing import Dict

import matplotlib.pyplot as plt
import numpy as np
import torch
from scipy.stats import spearmanr
from sklearn.metrics import r2_score
from torch_geometric.data import Data
from tqdm import tqdm

import barostat_parameters
from barostat_utils import (
    estimate_initial_box_vel_y,
    estimate_initial_box_vel_y_accurate,
    update_box_y_thermodynamic,
)
from graph_utils import prepare_traj
from pressure import (
    compute_per_particle_forces,
    compute_total_stress,
)
from simulator_model import Model as VelocityModel
from training_utils import (
    ModelInputs,
    freeze_normalizer,
    huber_loss,
    load_data_from_paths,
)
from utils import (
    build_velocity_graph_correction,
    calc_p_ratio_box_tensor,
    get_correct_edge_attr,
    get_rollout,
)


### Load Data

In [ ]:
dataset_type = "node_optimized" # "node_optimized" | "stiffness_optimized"
if dataset_type not in {"node_optimized", "stiffness_optimized"}:
    raise ValueError(f"Incorrect data type, expected 'node_optimized' or 'stiffness_optimized', got {dataset_type}. ")

data_dir = os.path.join("./demo_data", f"data_{dataset_type}_40sims.pt")
data = torch.load(data_dir, weights_only=False)

for data_type, sims in data.items():
    prepared = []
    for sim in tqdm(sims, desc=f"{data_type:<5}"):
        prepared_sim = prepare_traj(sim, calc_angles=False)
        prepared.append(prepared_sim)
    data[data_type] = prepared

print(f"Train data: {len(data['train'])} sims.")
print(f"FT data:    {len(data['ft'])} sims.")
print(f"Val data:   {len(data['val'])} sims.")
print(f"Test data:  {len(data['test'])} sims.")


#### Show $\nu$ distribution

In [ ]:
def visualize_nu_disribution(data: Dict):
    ps = {}
    all_values = []

    for data_type, sims in data.items():
        ps[data_type] = [calc_p_ratio_box_tensor(sim).item() for sim in sims]
        all_values.extend(ps[data_type])

    min_val = min(all_values)
    max_val = max(all_values)
    common_bins = np.linspace(min_val, max_val, 30) 

    for data_type, values in ps.items():
        plt.hist(
        values, 
        bins=common_bins, 
        edgecolor='black', 
        alpha=0.6, 
        label=f"{data_type} data"
    )

    plt.title("$\\nu$ distribution")
    plt.legend()
    plt.show()

visualize_nu_disribution(data)


### Training GNN simulator

#### Initiaize Velocity simulator

In [ ]:
mp_layers = 2
mlp = 3
hidden_size = 128
history = 3
device = "cuda"

init_graph = build_velocity_graph_correction(
    input_graphs=[data['train'][0][i].cpu().detach() for i in range(history + 1)],
    total_velocity=False,
    panic_at_positions=False
).to(device)

gnn_simulator = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)


#### One-step Training loop

In [ ]:
model_save_directory = f"./trained_models/{dataset_type}/OST"
if not os.path.exists(model_save_directory):
    os.makedirs(model_save_directory, exist_ok=True)

epochs = 100
freeze_norm_epoch = 5
train_sims = 100
val_sims = 20
train_limit = 15
accumulation_steps = 10
learning_rate = 1e-3
gamma = 0.995

optimizer = torch.optim.Adam(gnn_simulator.parameters(), lr=learning_rate, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)

optimizer.zero_grad()
for epoch in range(epochs):
    t_start = time.perf_counter()

    if epoch == freeze_norm_epoch:
        gnn_simulator.node_normalizer.frozen = True
        gnn_simulator.edge_normalizer.frozen = True
        gnn_simulator.output_normalizer.frozen = True

    total_acc_loss = 0
    total_val_loss = 0
    total_val_pos_mse = 0
    train_samples = 0
    val_samples = 0

    gnn_simulator.train()
    for sim in data['train'][:train_sims]:
        starting_points = [i for i in range(train_limit)]

        for i, idx in enumerate(starting_points):
            indices = [step + idx for step in range(history + 1)]
            target_idx = history + 1 + idx


            input_graphs_raw = [sim[k].detach().cpu() for k in indices]

            # Construct input graph
            input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device)

            # Construct ModelInputs
            model_inputs = ModelInputs(
                input_graphs_raw[-2].to(device) if history > 0 else input_graphs_raw[-1].to(device),
                input_graphs_raw[-1].to(device),
                sim[target_idx].to(device)
            )

            # Forward and Loss
            model_output = gnn_simulator(input_graph, is_training=True)
            acc_loss = huber_loss(gnn_simulator, model_output, model_inputs, is_training=True)

            # Backward
            loss_for_backward = acc_loss / accumulation_steps
            loss_for_backward.backward()

            # Optimization
            if (i + 1) % accumulation_steps == 0:
                torch.nn.utils.clip_grad_norm_(gnn_simulator.parameters(), max_norm=1.0)
                optimizer.step()
                optimizer.zero_grad()

            total_acc_loss += acc_loss.item()
            train_samples += 1

        optimizer.step()
        optimizer.zero_grad()

    # Validation
    with torch.no_grad():
        gnn_simulator.eval()
        for val_sim in data['val'][:val_sims]:
            
            starting_points = [i for i in range(train_limit)]

            for idx in starting_points:
                indices = [step + idx for step in range(history + 1)]
                target_idx = history + 1 + idx

                input_graphs_raw = [val_sim[k].detach().cpu() for k in indices]

                input_graph = build_velocity_graph_correction(input_graphs_raw, panic_at_positions=False).to(device) 

                val_inputs = ModelInputs(
                    input_graphs_raw[-2].to(device) if history > 0 else input_graphs_raw[-1].to(device),
                    input_graphs_raw[-1].to(device),
                    val_sim[target_idx].to(device),
                )

                # Forward and Loss
                model_output = gnn_simulator(input_graph, is_training=False)
                val_loss = huber_loss(gnn_simulator, model_output, val_inputs, is_training=False)

                # Update to next state and check position MSE
                pred_graph = gnn_simulator.update(val_inputs, model_output)
                pos_mse = torch.nn.functional.mse_loss(pred_graph.pos, val_sim[target_idx].to(device).pos)

                total_val_loss += val_loss.item()
                total_val_pos_mse += pos_mse.item()
                val_samples += 1

    lr_scheduler.step()

    # Statistics
    avg_train_loss = total_acc_loss / train_samples
    avg_val_loss = total_val_loss / val_samples
    avg_val_pos_mse = total_val_pos_mse / val_samples

    # Save model
    if epoch % 1 == 0:
        gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

    t_stop = time.perf_counter()
    print(
        f"Epoch {epoch + 1:>3} | "
        f"Train Loss: {avg_train_loss:.3e} | "
        f"Val Loss: {avg_val_loss:.3e} | "
        f"Val Pos MSE: {avg_val_pos_mse:.3e} | "
        f"Time: {t_stop - t_start:.2f} s"
    )


#### Multi-step training loop

In [ ]:
model_save_directory = f"./trained_models/{dataset_type}/MST"
if not os.path.exists(model_save_directory):
    os.makedirs(model_save_directory, exist_ok=True)

epochs = 100
train_limit: int = 15
max_rollout_steps: int = 10
fresh: bool = True
freeze_norm_epoch: int = 5
learning_rate: float = 1e-3
gamma = 0.995

barostat_config = barostat_parameters.node_optimizated if dataset_type == "node_optimized" else barostat_parameters.stiff_optimized

gnn_simulator.train()
params = filter(lambda p: p.requires_grad, gnn_simulator.parameters())
optimizer = torch.optim.Adam(params, lr=learning_rate, weight_decay=0.0)
lr_scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma, last_epoch=-1)


optimizer.zero_grad()
for epoch in range(epochs):
    t_start = time.perf_counter()

    if fresh:
        if epoch < 10:
            rollout_steps = 1
        elif epoch >= 10 and epoch < 20:
            rollout_steps = 2
        elif epoch >= 20 and epoch < 30:
            rollout_steps = 3
        elif epoch >= 30 and epoch < 40:
            rollout_steps = 5
        elif epoch >= 40 and epoch < 50:
            rollout_steps = 8
        else:
            rollout_steps = max_rollout_steps

    # Trackers
    total_acc_loss = 0
    train_samples = 0

    # Freeze normalizers
    if epoch == freeze_norm_epoch:
        gnn_simulator.node_normalizer.frozen = True
        gnn_simulator.edge_normalizer.frozen = True
        gnn_simulator.output_normalizer.frozen = True

    for sim in data['train']:
        # Get equilibrium bond lengths
        r0 = sim[0].edge_attr[:, -2]
        
        starting_points = [i for i in range(train_limit)]
        dump_period = barostat_config["default_skip"]

        for start_idx in starting_points:
            optimizer.zero_grad()

            indices = [step + start_idx for step in range(history + 1)]
            current_window_graphs = [sim[k].detach().to(device) for k in indices]

            b0 = current_window_graphs[-2].box_tensor[0]
            b1 = current_window_graphs[-1].box_tensor[0]
            box_compression_factor = b1 / b0

            if len(current_window_graphs) < 3:
                current_box_vel_y = estimate_initial_box_vel_y(
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    dump_period * barostat_config["dt"],
                )
            elif len(current_window_graphs) >= 3:
                current_box_vel_y = estimate_initial_box_vel_y_accurate(
                    current_window_graphs[-3],
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    dump_period * barostat_config["dt"],
                )
            else:
                raise Exception(f"Window size is too small : {len(current_window_graphs)}")


            rollout_loss = 0
            for step in range(rollout_steps):
                target_idx = history + 1 + start_idx + step
                target_graph = sim[target_idx].to(device)

                input_graph = build_velocity_graph_correction(current_window_graphs).to(device)

                model_inputs = ModelInputs(
                    current_window_graphs[-2],
                    current_window_graphs[-1],
                    target_graph,
                )

                model_output = gnn_simulator(input_graph, is_training=True)
                pred_graph_next = gnn_simulator.update(model_inputs, model_output)

                step_loss = huber_loss(gnn_simulator, model_output, model_inputs, is_training=True)
                rollout_loss += step_loss

                dt = barostat_config["dt"]  # lammps dt
                W_y = barostat_config["C_coupling"] * pred_graph_next.num_nodes * ((dump_period * dt) ** 2)
                damping = barostat_config["damping"] * pred_graph_next.num_nodes * (dump_period * dt)

                new_lx = pred_graph_next.box_tensor[0] * box_compression_factor
                new_ly, new_vel_y = update_box_y_thermodynamic(
                    positions=pred_graph_next.pos,
                    edge_index=model_inputs.cur_graph.edge_index,
                    edge_attr=model_inputs.cur_graph.edge_attr,
                    current_box=model_inputs.cur_graph.box_tensor,
                    r0=r0.to(pred_graph_next.pos.device),
                    box_vel_y=current_box_vel_y,  # Use ESTIMATED velocity
                    W_y=W_y,
                    damping=damping,
                    stride_dt=dump_period * dt,
                    target_pressure=barostat_config["target_pressure"],
                    temperature=barostat_config["temperature"],
                )

                new_box_tensor = torch.stack([new_lx, new_ly])
                current_box_vel_y = new_vel_y

                # Add new box and update edge_attr
                pred_graph_next.box_tensor = new_box_tensor
                pred_graph_next.edge_attr = get_correct_edge_attr(
                    pred_graph_next,
                    recompute_stiff=False,
                    panic_at_nontensor_box=True,
                )
                pred_graph_next.forces = compute_per_particle_forces(pred_graph_next, r0=r0.to(pred_graph_next.pos.device))

                pred_graph_next_detached = pred_graph_next.detach()

                # Update window: shift left, append new prediction
                current_window_graphs.pop(0)
                current_window_graphs.append(pred_graph_next_detached)

            final_loss = rollout_loss / rollout_steps
            final_loss.backward()

            # Clip gradients (essential for GNNs in physics)
            torch.nn.utils.clip_grad_norm_(gnn_simulator.parameters(), max_norm=1.0)
            optimizer.step()

            total_acc_loss += final_loss.item()
            train_samples += 1

    gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"checkpoint_epoch_{epoch}.pt"))

    total_acc_loss /= train_samples
    lr_scheduler.step()
    t_stop = time.perf_counter()
    print(f"Epoch {epoch:<3} | steps: {rollout_steps:<2} | loss: {total_acc_loss:.4e} | {t_stop - t_start:.2f} s.")


#### Save trained simulator model

In [ ]:
gnn_simulator.save_checkpoint(os.path.join(model_save_directory, f"model_h{history}_nl{mp_layers}_mlp{mlp}_epochs{epochs}.pt"))

### Testing models

#### Load saved models

In [ ]:
mp_layers = 2
mlp = 3
hidden_size = 128
history = 3
device = "cuda"

init_graph = Data(x=torch.ones((100, history*2)), edge_attr=torch.ones((100, 4)))

models = {}

model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
model.load_checkpoint(f"./trained_models/{dataset_type}/OST/model_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
model = freeze_normalizer(model)
models[f"h{3} ost"] = model

model: VelocityModel = VelocityModel(init_graph, hidden_size, mp_layers, mlp).to(device)
model.load_checkpoint(f"./trained_models/{dataset_type}/MST/model_h{history}_nl{mp_layers}_mlp{mlp}_epochs100.pt")
model = freeze_normalizer(model)
models[f"h{3} mst"] = model

models.keys()

#### Test model performance

In [ ]:
factors = [(sim[4].box_tensor[0]/sim[3].box_tensor[0]).item() for sim in data["test"]]
mean_factor = sum(factors)/len(factors)
print(mean_factor)

In [ ]:
num_steps = 50
model_results = {}
test_models = models

barostat_config = barostat_parameters.node_optimizated if dataset_type == "node_optimized" else barostat_parameters.stiff_optimized

name_len = max([len(n) for n in test_models.keys()])+1 

for model_name, model in test_models.items():
    model.eval()

    history = int(model_name[1])
    target_idx = num_steps + history + 1
    results = {
        "gt_box_velocity": [],
        "pred_box_velocity": [],
        "final_mse": [],
        "mse": [],
        "est_p": [],
        "pred_p": [],
        "gt_est_p": [],
        "gt_p": [],
        "gt_box": [],
        "pred_box": [],
        "gt_forces": [],
        "pred_forces": [],
        "gt_pressure": [],
        "pred_pressure": [],
    }

    with torch.no_grad():
        for val_sim in tqdm(data['val'][:30] + data['test'][:30] + data['train'][:30] + data['ft'][:30], desc=f"Model {model_name:<{name_len}}"):

            # This block computes dumping period (N MD steps in 1 dump step)        
            sim_strain = (val_sim[1].box.x - val_sim[-1].box.x) / val_sim[0].box.x
            assumed_rollout_length = int(sim_strain / 1e-5 / 0.01)
            dump_period = int(assumed_rollout_length / len(val_sim)) + 1
                        
            input_graphs = [g.cpu().detach() for g in val_sim[: history + 1]]

            rollout = get_rollout(
                input_graphs=input_graphs,
                gnn_simulator=model,
                gnn_history=history,
                num_steps=num_steps,
                barostat_config=barostat_config,
                rescale=False,
                device="cuda"
            )

            # Compare Predicted Position vs Ground Truth Position
            pos_mse = torch.nn.functional.mse_loss(rollout[-1].pos.cpu(), val_sim[target_idx].to(device).pos.cpu()).item()
            results["final_mse"].append(pos_mse)
            pos_mse = [torch.nn.functional.mse_loss(rollout[i].pos.cpu(), val_sim[i].to(device).pos.cpu()).item() for i in range(len(rollout))]
            results["mse"].append(pos_mse)
            
            pred_p = calc_p_ratio_box_tensor(rollout).item()
            results["pred_p"].append(pred_p)
            gt_p = calc_p_ratio_box_tensor(val_sim[:target_idx]).item()
            results["gt_p"].append(gt_p)

            pred_box = [g.box_tensor.cpu() for g in rollout]
            results["pred_box"].append(pred_box)
            gt_box = [g.box_tensor.cpu() for g in val_sim[: len(rollout)]]
            results["gt_box"].append(gt_box)

            gt_pressure = torch.stack([compute_total_stress(g, r0=input_graphs[0].edge_attr[:, -2].to(g.x.device), temperature=1e-7).cpu() for g in val_sim[: len(rollout)]], dim=0)
            results['gt_pressure'].append(gt_pressure)
            pred_pressure = torch.stack([compute_total_stress(g, r0=input_graphs[0].edge_attr[:, -2].to(g.x.device), temperature=1e-7).cpu() for g in rollout], dim=0)
            results["pred_pressure"].append(pred_pressure)

    model_results[model_name] = results

In [ ]:
params = {
    'font.size': 8,                 # Base font size
    'axes.labelsize': 8,            # Axis labels (e.g., nu_gt)
    'axes.titlesize': 8,            # Subplot titles
    'xtick.labelsize': 7,           # Tick numbers
    'ytick.labelsize': 7,   
    'legend.fontsize': 6,           # Make legend smaller to fit
    'lines.markersize': 4,          # Reduce scatter dot size
    'figure.figsize': (3.33, 3.33), # Your target size
    'figure.dpi': 200,              # High DPI for clear viewing
    'font.family': 'serif',         # Matches most LaTeX/Paper fonts
}
plt.rcParams.update(params)


In [ ]:
fig, ax = plt.subplots(1, 1, layout="constrained")

line = (min(results["gt_p"]) - 0.05, max(results["gt_p"]) + 0.05)
ax.plot(line, line, color="black", linewidth=1, linestyle='--')

for model_name, results in model_results.items():

    r2 = r2_score(results['gt_p'], results['pred_p'])
    res = spearmanr(results["gt_p"], results["pred_p"])
    sp = res.statistic
    
    label_text = f"Model {model_name}: $R^2={r2:.3f}$, SP={sp:.3f}"
    ax.scatter(results["gt_p"], results["pred_p"], label=label_text)
    
ax.legend(frameon=False, loc='best')
ax.set_xlabel(r"$\nu_{gt}$")
ax.set_ylabel(r"$\nu_{pred}$")


plt.show()

#### MSE as a function of strain

In [ ]:
rand_sim = random.randint(0, len(results['mse'])-1)
for model_name, results in model_results.items():
    plt.plot(results["mse"][rand_sim], label=f"Model {model_name}")

plt.legend()
plt.yscale('log')
plt.title(f"Sim {rand_sim}")
plt.xlabel('Rollout step')
plt.ylabel('Position MSE')
plt.show()

In [ ]:
rand = random.randint(0, len(results) - 1)

fig, ax = plt.subplots(2, 2, layout="constrained", sharex=True)

box_y_true = [model_results["h3 ost"]["gt_box"][rand][i][1].item() for i in range(1, len(rollout) - 1)]
box_x_true = [model_results["h3 ost"]["gt_box"][rand][i][0].item() for i in range(1, len(rollout) - 1)]

pressure_gt = results["gt_pressure"][rand]

ax[0][0].plot(box_y_true, label="$L_y$ GT")
ax[1][0].plot(box_x_true, label="$L_x$ GT")
ax[0][1].plot(pressure_gt[:, 1], label="$P_{{yy}}$ GT")
ax[1][1].plot(pressure_gt[:, 0], label="$P_{{xx}}$ GT")


for model_name, results in model_results.items():


    box_y_roll = [results["pred_box"][rand][i][1].item() for i in range(1, len(rollout) - 1)]
    box_x_roll = [results["pred_box"][rand][i][0].item() for i in range(1, len(rollout) - 1)]

    pressure_roll = results["pred_pressure"][rand]

    ax[0][0].plot(box_y_roll, label=f"$\\hat{{L_y}}$ {model_name.split()[1]}")
    ax[0][0].legend()

    ax[1][0].plot(box_x_roll, label="$L_x$ rollout")
    ax[1][0].legend()

    ax[0][1].plot(pressure_roll[:, 1], label=f"$P_{{yy}}$ {model_name.split()[1]}")
    ax[0][1].legend()
    
    ax[1][1].plot(pressure_roll[:, 0], label=f"$P_{{xx}}$ {model_name.split()[1]}")
    ax[1][1].legend()

plt.show()